# 01 — Data Exploration: Stanford Alpaca Dataset

**Project**: LLM Fine-Tuning with QLoRA  
**Notebook**: 1 of 6  
**GPU required**: ❌ No — pure data exploration  
**Runtime**: ~5 minutes

---

## What This Notebook Does

Before touching the model, we understand the data we're training on.
This notebook:
1. Sets up the environment and clones the project repo
2. Loads the Stanford Alpaca dataset from HuggingFace Hub
3. Explores its structure, statistics, and example content
4. Applies the prompt template and shows what the model will actually see
5. Creates and saves the train/test split we'll use in all future notebooks

## Why Start Here?

> "In machine learning, your model is only as good as your data."
> Understanding the data is step one of every serious ML project.

**The Stanford Alpaca Dataset**  
Created by researchers at Stanford in 2023. It contains 52,000 instruction-following
examples generated using GPT-3.5-turbo. We use the cleaned version (`yahma/alpaca-cleaned`)
which removed ~1,700 noisy examples from the original.

Each example has three fields:
- `instruction`: what the model should do
- `input`: optional context (empty for most examples)
- `output`: the correct response

---
## ⚙️ Cell 1: Environment Setup

First, we install dependencies and clone our project repo from GitHub.
Run this cell first — it only needs to run once per Colab session.

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
# We install from our requirements.txt so versions are pinned and consistent
!pip install -q datasets transformers matplotlib pandas wordcloud

print("✅ Dependencies installed")

In [ ]:
# ── Clone project repo from GitHub ────────────────────────────────────────
import os

GITHUB_USERNAME = "lambda06"
REPO_NAME = "llm-finetuning-peft"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print(f"✅ Repo cloned: {REPO_NAME}/")
else:
    !cd {REPO_NAME} && git pull
    print(f"✅ Repo already exists — pulled latest changes")

# Add src/ to Python path so we can import our modules
import sys
sys.path.insert(0, f"/content/{REPO_NAME}/src")
print(f"✅ Added src/ to Python path")

---
## 📥 Cell 2: Load the Dataset

We load `yahma/alpaca-cleaned` from HuggingFace Hub using the `datasets` library.

**What is HuggingFace Hub?**  
Think of it as GitHub but for ML models and datasets. Anyone can publish and download
datasets with a single line of code. No manual downloading, no unzipping — just `load_dataset()`.

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the full dataset (downloads ~25 MB, cached after first run)
print("📥 Loading yahma/alpaca-cleaned from HuggingFace Hub...")
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

print(f"\n📊 Dataset loaded!")
print(f"   Total examples : {len(dataset):,}")
print(f"   Features       : {list(dataset.features.keys())}")
print(f"   Dataset type   : {type(dataset)}")

# Convert to pandas for easier exploration
df = dataset.to_pandas()
print(f"\n   Shape (rows, cols): {df.shape}")

---
## 🔍 Cell 3: Understand the Structure

Let's look at what a few examples actually contain.

In [ ]:
# Show first 5 examples in a readable table
print("=" * 80)
print("FIRST 5 EXAMPLES")
print("=" * 80)

for i in range(5):
    ex = dataset[i]
    print(f"\nExample {i+1}:")
    print(f"  INSTRUCTION : {ex['instruction'][:100]}..." if len(ex['instruction']) > 100 else f"  INSTRUCTION : {ex['instruction']}")
    print(f"  INPUT       : {ex['input'][:80]}..." if len(ex['input']) > 80 else f"  INPUT       : '{ex['input']}'")
    print(f"  OUTPUT      : {ex['output'][:100]}..." if len(ex['output']) > 100 else f"  OUTPUT      : {ex['output']}")
    print("-" * 80)

In [ ]:
# ── Check: how many examples have an 'input' field vs instruction-only? ───
has_input = df[df['input'].str.strip() != '']
no_input  = df[df['input'].str.strip() == '']

print("📊 Input field distribution:")
print(f"   With extra input context : {len(has_input):,} ({100*len(has_input)/len(df):.1f}%)")
print(f"   Instruction only         : {len(no_input):,} ({100*len(no_input)/len(df):.1f}%)")
print()
print("→ This tells us which prompt template to use for each example.")
print("→ Our data_utils.py handles both cases automatically.")

---
## 📏 Cell 4: Text Length Analysis

Understanding text lengths helps us set `max_seq_length` in the config.  
If we set it too low, long examples get truncated and we lose information.  
If we set it too high, we waste GPU memory on padding.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Compute character lengths
df['instruction_len'] = df['instruction'].str.len()
df['input_len']       = df['input'].str.len()
df['output_len']      = df['output'].str.len()
df['total_len']       = df['instruction_len'] + df['input_len'] + df['output_len']

print("📏 Text Length Statistics (characters):")
print(df[['instruction_len', 'output_len', 'total_len']].describe().round(1).to_string())

# Plot distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Alpaca Dataset — Text Length Distributions', fontsize=14, fontweight='bold')

colors = ['#4F46E5', '#10B981', '#F59E0B']
cols   = ['instruction_len', 'output_len', 'total_len']
titles = ['Instruction Length', 'Output Length', 'Total Length']

for ax, col, title, color in zip(axes, cols, titles, colors):
    ax.hist(df[col].clip(upper=df[col].quantile(0.99)), bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Characters')
    ax.set_ylabel('Count')
    ax.axvline(df[col].median(), color='red', linestyle='--', linewidth=1.5, label=f'Median: {df[col].median():.0f}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('length_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\n💡 Our max_seq_length=512 tokens ≈ ~2,000 characters")
print(f"   Median total length: {df['total_len'].median():.0f} characters → well within limit ✅")

---
## 📝 Cell 5: The Prompt Template

This is critical — the model doesn't see `instruction`, `input`, `output` separately.
We pack everything into **one string** using a fixed template.

The model learns: *"when I see `### Instruction:`, I should generate after `### Response:`"*

In [ ]:
# Import our prompt formatting function from src/
from data_utils import format_prompt, PROMPT_WITH_INPUT, PROMPT_WITHOUT_INPUT

# ── Show the two templates ─────────────────────────────────────────────────
print("=" * 60)
print("TEMPLATE 1 — Instruction only (no input context)")
print("=" * 60)
print(PROMPT_WITHOUT_INPUT.format(
    instruction="List 3 benefits of exercise.",
    output="1. Improves cardiovascular health\n2. Boosts mood\n3. Increases strength"
))

print()
print("=" * 60)
print("TEMPLATE 2 — With extra input context")
print("=" * 60)
print(PROMPT_WITH_INPUT.format(
    instruction="Summarize the following text in one sentence.",
    input="The sun is a star at the center of our solar system. It provides light and heat to Earth.",
    output="The sun is a central star that provides essential light and heat to Earth."
))

In [ ]:
# ── Apply formatting to the full dataset ──────────────────────────────────
print("📝 Applying prompt template to all examples...")
formatted_dataset = dataset.map(
    format_prompt,
    remove_columns=['instruction', 'input', 'output']
)

print(f"\n✅ Done! Dataset now has columns: {list(formatted_dataset.features.keys())}")
print(f"\n📌 Example formatted prompt:")
print("-" * 60)
print(formatted_dataset[0]['text'])
print("-" * 60)

---
## ✂️ Cell 6: Create Train / Test Split

We take a subset for fast training on a free T4, then split into train and test sets.  
The test set is **never seen during training** — it's our unbiased evaluation set.

**Why `seed=42`?**  
Setting a random seed makes the shuffle deterministic. Run this notebook 100 times
and you always get the same split. This is essential for reproducibility — a core
principle of scientific ML work.

In [ ]:
from data_utils import load_alpaca_dataset

# Load with our utility function (handles shuffle + split internally)
train_dataset, test_dataset = load_alpaca_dataset(
    train_size=2000,
    test_size=200,
    seed=42
)

print(f"\n📊 Split Summary:")
print(f"   Training   : {len(train_dataset):,} examples")
print(f"   Test       : {len(test_dataset):,} examples")
print(f"   Total used : {len(train_dataset) + len(test_dataset):,} of {len(dataset):,} available")
print(f"   Unused     : {len(dataset) - len(train_dataset) - len(test_dataset):,} examples")

In [ ]:
# ── Verify no overlap between train and test ───────────────────────────────
train_texts = set(train_dataset['text'])
test_texts  = set(test_dataset['text'])
overlap = train_texts & test_texts

print(f"🔍 Data leakage check:")
print(f"   Overlapping examples between train and test: {len(overlap)}")
if len(overlap) == 0:
    print(f"   ✅ No overlap — test set is completely unseen during training")
else:
    print(f"   ⚠️  WARNING: Found {len(overlap)} overlapping examples!")

---
## 📊 Cell 7: Dataset Statistics Summary

A final summary view — the kind of thing you'd put in a research report or README.

In [ ]:
# ── Formatted summary table ────────────────────────────────────────────────
import textwrap

# Compute token-level stats using a rough estimate (1 token ≈ 4 characters)
train_df = train_dataset.to_pandas()
train_df['approx_tokens'] = train_df['text'].str.len() / 4

print("=" * 60)
print("  DATASET SUMMARY — yahma/alpaca-cleaned (subset)")
print("=" * 60)
print(f"  {'Source dataset':<28}: yahma/alpaca-cleaned")
print(f"  {'Full dataset size':<28}: 51,760 examples")
print(f"  {'Training examples':<28}: {len(train_dataset):,}")
print(f"  {'Test examples':<28}: {len(test_dataset):,}")
print(f"  {'Random seed':<28}: 42")
print(f"  {'Avg tokens per example':<28}: {train_df['approx_tokens'].mean():.0f} (approx)")
print(f"  {'Max seq length (config)':<28}: 512 tokens")
print(f"  {'Examples with input context':<28}: ~40%")
print(f"  {'Instruction-only examples':<28}: ~60%")
print("=" * 60)

print("\n💡 Key takeaway:")
print("   Average example is ~100 tokens — well within our 512-token limit.")
print("   No truncation expected for the vast majority of training examples.")

---
## ✅ Summary

In this notebook you:

| Step | What happened |
|---|---|
| Loaded dataset | `yahma/alpaca-cleaned` — 51,760 instruction-following examples |
| Explored structure | 3 fields: `instruction`, `input`, `output` |
| Checked lengths | Most examples fit comfortably in 512 tokens |
| Applied template | Packed fields into one `text` string with `### Instruction:` headers |
| Created split | 2,000 train / 200 test, seed=42, zero overlap |

### Key Concepts Covered
- **HuggingFace `datasets`**: Load and manipulate datasets with `.map()`, `.shuffle()`, `.select()`
- **Prompt template**: Why format matters — the model learns the template pattern
- **Train/test split**: Why we never let the model see test data during training
- **Reproducibility**: Why `seed=42` matters in ML experiments

---
### ▶️ Next: `02_baseline_inference.ipynb`
We load TinyLlama and run these same prompts through it **before fine-tuning**.  
These are our baseline outputs — saved so we can compare after training.